# Tournament simulation

## 1. One match
I simulate a match many times from the Poisson expected goals. In knockout games, a level score goes to extra time (one third of the expected goals), then to a penalty shootout.
Shootouts are a coin flip: since 2000, the higher-rated team wins 53.8% of 355 shootouts, which is within 1.4 standard errors of 50%.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd

from src.database import query
from src.models.poisson import PoissonModel
from src.simulation import simulate_knockout, simulate_scores

history = query("""
    SELECT date, home_team, away_team, home_score, away_score, neutral
    FROM clean_results
    ORDER BY date, match_id
""")
model = PoissonModel().fit(history, "2026-06-11")

In [2]:
final = pd.DataFrame({"home_team": ["Spain"], "away_team": ["Argentina"], "neutral": [True]})
home_goals, away_goals = model.expected_goals(final)
exact = model.predict_proba(final)[0]

rng = np.random.default_rng(42)
home, away = simulate_scores(home_goals[0], away_goals[0], 100_000, rng)
pd.DataFrame({
    "exact": exact,
    "simulated": [(home > away).mean(), (home == away).mean(), (home < away).mean()],
}, index=["Spain win", "draw", "Argentina win"]).round(3)

,exact,simulated
Spain win,0.281,0.282
draw,0.318,0.320
Argentina win,0.401,0.398


In [3]:
rng = np.random.default_rng(42)
knockout = simulate_knockout(home_goals[0], away_goals[0], 100_000, rng)
{name: round(values.mean(), 3) for name, values in knockout.items()}

{'home_wins': np.float64(0.433),
 'extra_time': np.float64(0.32),
 'shootout': np.float64(0.19)}

With 100,000 simulations, the simulated outcome shares match the exact Poisson probabilities within 0.3 points, the expected Monte Carlo error.
Before the 2026 World Cup, the Poisson model saw Argentina as the favourite in a final against Spain (40% vs 28% after 90 minutes), because of Argentina's very strong defence. In a knockout game, Spain would still go through 43% of the time: extra time in 32% of games, a shootout in 19%.
The actual final was 0-0 after 90 minutes and Spain won in extra time, one of the most likely paths under the model.

In [4]:
for as_of in ["2026-06-11", "2026-07-19"]:
    model_at_date = PoissonModel().fit(history, as_of)
    home_goals, away_goals = model_at_date.expected_goals(final)
    knockout = simulate_knockout(home_goals[0], away_goals[0], 100_000, np.random.default_rng(42))
    print(as_of, home_goals.round(2), away_goals.round(2),
          model_at_date.predict_proba(final).round(3), knockout["home_wins"].mean().round(3))

2026-06-11 [0.82] [1.05] [[0.281 0.318 0.401]] 0.433
2026-07-19 [0.95] [0.91] [[0.352 0.322 0.326]] 0.514


A tournament forecast uses the model fitted before the tournament for every match, even the final, five weeks later. A match-by-match forecast refits the model the day before each game. Refitted on 19 July with all World Cup results, the model makes Spain a slight favourite (35% vs 33%, 51% to win the final), close to the 0-0 and extra-time win that happened.

## 2. Group stage
I simulate the 12 groups 10,000 times with the model fitted on 11 June 2026. Ties are broken by goal difference, goals scored, then at random. The 8 best third-placed teams also qualify.
Host nations get the Poisson home advantage in their group matches (about 26% more goals).

In [5]:
from src.tournament import goal_rates, load_config, simulate_group_stage

config = load_config("world_cup_2026")
teams = [team for members in config["groups"].values() for team in members]
team_index = {team: i for i, team in enumerate(teams)}
rates, home_factor = goal_rates(model, teams)
n_sims = 10_000
positions, best_thirds = simulate_group_stage(config, team_index, rates, home_factor, n_sims, np.random.default_rng(42))

In [6]:
def share(team_ids):
    return np.bincount(team_ids.ravel(), minlength=len(teams)) / n_sims

first = sum(share(positions[f"1{letter}"]) for letter in config["groups"])
top_two = first + sum(share(positions[f"2{letter}"]) for letter in config["groups"])
qualified_thirds = sum(
    share(positions[f"3{letter}"][(best_thirds == letter).any(axis=1)]) for letter in config["groups"]
)
group_stage = pd.DataFrame({
    "group": [letter for letter, members in config["groups"].items() for _ in members],
    "first": first,
    "top_two": top_two,
    "qualified": top_two + qualified_thirds,
}, index=teams)
group_stage.sort_values("qualified", ascending=False).round(3)

,group,first,top_two,qualified
Spain,H,0.745,0.960,0.989
Brazil,C,0.595,0.892,0.984
England,L,0.666,0.936,0.984
Argentina,J,0.690,0.910,0.980
Germany,E,0.529,0.829,0.979
Switzerland,B,0.549,0.891,0.968
Belgium,G,0.610,0.860,0.955
Morocco,C,0.319,0.765,0.941
Canada,B,0.390,0.829,0.937
Ecuador,E,0.312,0.712,0.937


## 3. Knockout stage
The 8 best third-placed teams fill the 8 third-place slots of the round of 32, following the FIFA list of allowed groups for each slot (backtracking search, cached for each of the 495 combinations). Each knockout game uses extra time and shootouts, with home advantage for a host playing in its own country.

In [7]:
from src.tournament import simulate_tournament

forecast = simulate_tournament(config, model, n_sims=10_000, seed=42)
forecast.sum().round(2)

round_of_32      32.0
round_of_16      16.0
quarter_final     8.0
semi_final        4.0
final             2.0
champion          1.0
dtype: float64

In [8]:
forecast.sort_values("champion", ascending=False).head(15).round(3)

,round_of_32,round_of_16,quarter_final,semi_final,final,champion
Argentina,0.980,0.684,0.530,0.382,0.262,0.177
Spain,0.989,0.642,0.437,0.317,0.210,0.125
England,0.984,0.690,0.458,0.276,0.155,0.089
Brazil,0.984,0.646,0.434,0.274,0.153,0.080
Portugal,0.918,0.632,0.378,0.208,0.118,0.062
France,0.932,0.629,0.385,0.226,0.121,0.060
Colombia,0.908,0.602,0.346,0.191,0.099,0.048
Netherlands,0.902,0.488,0.316,0.172,0.084,0.041
Morocco,0.941,0.520,0.311,0.171,0.083,0.041
Germany,0.979,0.666,0.353,0.193,0.094,0.041


Every column sums to the number of teams in that round (32, 16, 8, 4, 2, 1), so no team is lost or counted twice.
Before the tournament, Argentina was the favourite (17.7% to win), ahead of Spain (12.5%), England and Brazil. Even the favourite has less than one chance in five: winning takes five knockout games.

## 4. Backtest of the 2026 World Cup
The 2026 World Cup is part of the test set. I evaluate it now because the model is frozen: Poisson and its settings were chosen on validation, and I will not change them after this result.
For each round, I compare the forecast probability of reaching it with what happened (1 or 0), using the Brier score. The naive baseline gives every team the same probability (32/48 for the round of 32, and so on).

In [9]:
from src.tournament import actual_progress

actual = actual_progress(config, "2026-07-19")
actual.sum()

round_of_32      32
round_of_16      16
quarter_final     8
semi_final        4
final             2
champion          1
dtype: int64

In [10]:
rows = []
for round_name in forecast.columns:
    predicted, reached = forecast[round_name], actual[round_name]
    rows.append({
        "round": round_name,
        "brier": ((predicted - reached) ** 2).mean(),
        "brier_naive": ((reached.mean() - reached) ** 2).mean(),
        "mean_p_reached": predicted[reached == 1].mean(),
        "mean_p_others": predicted[reached == 0].mean(),
    })
pd.DataFrame(rows).round(3)

,round,brier,brier_naive,mean_p_reached,mean_p_others
0,round_of_32,0.146,0.222,0.785,0.430
1,round_of_16,0.131,0.222,0.543,0.229
2,quarter_final,0.090,0.139,0.378,0.124
3,semi_final,0.049,0.076,0.300,0.064
4,final,0.027,0.040,0.236,0.033
5,champion,0.017,0.020,0.125,0.019


In [11]:
actual_quarter_finalists = actual.index[actual["quarter_final"] == 1]
forecast.loc[actual_quarter_finalists, ["quarter_final", "semi_final", "final", "champion"]].round(3)

,quarter_final,semi_final,final,champion
Switzerland,0.284,0.120,0.050,0.022
Morocco,0.311,0.171,0.083,0.041
Belgium,0.363,0.169,0.083,0.035
Spain,0.437,0.317,0.210,0.125
France,0.385,0.226,0.121,0.060
Norway,0.256,0.120,0.052,0.023
Argentina,0.530,0.382,0.262,0.177
England,0.458,0.276,0.155,0.089


The forecast beats the naive baseline in every round, with a Brier score 25 to 40% lower up to the final. Teams that reached a round had a much higher forecast probability than the others (30% vs 6% for the semi-finals).
Among the 8 actual quarter-finalists, the 4 semi-finalists had the 4 highest semi-final probabilities, and the final was played by the two teams with the highest final probabilities (Argentina 26%, Spain 21%). Spain won the title as the second favourite (12.5%).
The gain is small for the champion (0.017 vs 0.020): one tournament is a single draw, and judging a title forecast needs many tournaments.
Main misses: Uruguay out in the group stage (89% to qualify), Brazil beaten by Norway in the round of 16.